In [1]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from pathlib import Path
from time import perf_counter
import inspect
import re
import urllib.request
import zipfile

import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageDraw
from IPython.display import display
from tqdm.auto import tqdm

from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from ultralytics import SAM

In [3]:
PROJECT = Path("/content/drive/MyDrive/Vision/vision_unit_04")
MANIFEST = PROJECT / "outputs/evaluation_manifest.csv"
DATA = Path("/content/datasets/coco128-seg")

manifest = pd.read_csv(MANIFEST)
cal_names = manifest.loc[manifest.split == "calibration", "image_name"].tolist()
test_names = manifest.loc[manifest.split == "test", "image_name"].tolist()

print(f"Calibration: {len(cal_names)} images | Test: {len(test_names)} images")

Calibration: 20 images | Test: 20 images


In [5]:
CLASSES = {0: "person", 2: "car", 16: "dog", 41: "cup", 56: "chair"}
NAME_TO_ID = {name: cid for cid, name in CLASSES.items()}
PROMPTS = {
    "combined": ("chair. cup. dog. car. person.",),
    "separate": ("chair.", "cup.", "dog.", "car.", "person."),
}
BOX_THRESHOLD, TEXT_THRESHOLD = 0.35, 0.30
MODEL_ID = "IDEA-Research/grounding-dino-tiny"

In [6]:
processor = AutoProcessor.from_pretrained(MODEL_ID)
detector = AutoModelForZeroShotObjectDetection.from_pretrained(MODEL_ID)
detector = detector.to("cuda").float().eval()
sam = SAM("sam2.1_s.pt")

postprocess = processor.post_process_grounded_object_detection
box_argument = (
    "threshold" if "threshold" in inspect.signature(postprocess).parameters
    else "box_threshold"
)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/978 [00:00<?, ?it/s]

**Read Image and GT**

In [7]:
def load_sample(name):
    image = Image.open(DATA / "images/train2017" / name).convert("RGB")
    width, height = image.size
    
    label_path = DATA / "labels/train2017" / f"{Path(name).stem}.txt"
    gt = []

    for line in label_path.read_text().splitlines():
        if not line.strip():
            continue
        
        values = list(map(float, line.split()))
        cid = int(values[0])
        if cid not in CLASSES:
            continue
        
        points = np.array(values[1:]).reshape(-1, 2) * [width, height]
        points = np.clip(points, [0, 0], [width - 1, height - 1])
        canvas = Image.new("L", image.size, 0)
        
        ImageDraw.Draw(canvas).polygon([tuple(p) for p in points], fill=1)
        mask = np.asarray(canvas, dtype=bool)
        ys, xs = np.nonzero(mask)
        
        if len(xs):
            gt.append({
                "class_id": cid, "mask": mask,
                "box": np.array([xs.min(), ys.min(), xs.max() + 1, ys.max() + 1]),
            })
            
    return image, gt


**Text → boxes**

In [ ]:
@torch.inference_mode()
def detect(image, prompt, box_threshold, text_threshold):
    inputs = processor(
        images=image, text=prompt, return_tensors="pt"
    ).to("cuda")
    
    outputs = detector(**inputs)
    result = postprocess(
        outputs, inputs.input_ids,
        **{box_argument: box_threshold},
        text_threshold=text_threshold,
        target_sizes=[image.size[::-1]]
    )[0]
    
    labels = (
        result["text_labels"] if "text_labels" in result
        else result["labels"]
    )
    
    return {
        "boxes": result["boxes"].cpu().numpy().reshape(-1, 4),
        "scores": result["scores"].cpu().numpy(),
        "labels": list(labels)
    }

**Reusable grounded inference**

In [9]:
def grounded_predict(image, prompts, box_threshold=0.35, text_threshold=0.30):
    image = image.convert("RGB")
    torch.cuda.synchronize()
    start = perf_counter()
    parts = [detect(image, p, box_threshold, text_threshold) for p in prompts]
    
    prediction = {
        "boxes": np.concatenate([p["boxes"] for p in parts]),
        "scores": np.concatenate([p["scores"] for p in parts]),
        "labels": [label for p in parts for label in p["labels"]],
    }
    
    boxes = prediction["boxes"]
    if len(boxes):
        result = sam(
            image, bboxes=boxes.tolist(), device=0, imgsz=1024,
            retina_masks=True, multimask_output=False, verbose=False,
        )[0]
        masks = result.masks.data.cpu().numpy() > 0.5
    else:
        masks = np.zeros((0, image.height, image.width), dtype=bool)


    torch.cuda.synchronize()
    prediction["total_ms"] = (perf_counter() - start) * 1000
    prediction["masks"] = masks
    return prediction